In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('tagasiside_log.csv')
print(f'Kokku päringuid: {len(df)}')
df[['Aeg', 'Kasutaja päring', 'Hinnang', 'Veatüüp']]

## 1. Üldstatistika

In [ ]:
total = len(df)
good = (df['Hinnang'] == '👍 Hea').sum()
bad = (df['Hinnang'] == '👎 Halb').sum()

print(f'Kokku päringuid:   {total}')
print(f'Head vastused:     {good} ({good/total*100:.0f}%)')
print(f'Halvad vastused:   {bad} ({bad/total*100:.0f}%)')

## 2. Vigade analüüsi tabel

Vahesammud:
1. **Metaandmete filtreerimine** — filtrid on liiga karmid või valed
2. **RAG vektorotsing** — otsing leidis valed ained
3. **LLM genereerimine** — mudel hallutsineeris või ignoreeris konteksti

In [ ]:
bad_df = df[df['Hinnang'] == '👎 Halb'].copy()

# Lühendame veatüübi nimetused
def lyhenda(x):
    if 'Filtrid' in str(x):
        return 'Metaandmete filtreerimine'
    elif 'RAG' in str(x) or 'Otsing' in str(x):
        return 'RAG vektorotsing'
    elif 'LLM' in str(x) or 'hallutsineeris' in str(x):
        return 'LLM genereerimine'
    return str(x)

bad_df['Vahesamm'] = bad_df['Veatüüp'].apply(lyhenda)

# Vigade tabel
error_table = bad_df[['Aeg', 'Kasutaja päring', 'Filtrid', 'Vahesamm']].copy()
error_table.columns = ['Aeg', 'Päring', 'Kasutatud filtrid', 'Süüdi olev vahesamm']
error_table = error_table.reset_index(drop=True)
error_table.index += 1
print('=== VIGADE ANALÜÜSI TABEL ===')
error_table

## 3. Vigade jaotus vahesammude kaupa

In [ ]:
vahesammud = ['Metaandmete filtreerimine', 'RAG vektorotsing', 'LLM genereerimine']
counts = bad_df['Vahesamm'].value_counts().reindex(vahesammud, fill_value=0)

summary = pd.DataFrame({
    'Vahesamm': vahesammud,
    'Vigade arv': counts.values,
    'Protsent kõikidest vigastest päringutest': [f'{v/len(bad_df)*100:.0f}%' for v in counts.values]
})
summary = summary.set_index('Vahesamm')
print('=== VIGADE JAOTUS ===')
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Tulpdiagramm
colors = ['#e74c3c' if v > 0 else '#95a5a6' for v in counts.values]
axes[0].bar(vahesammud, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Vigade arv vahesammude kaupa', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Vigade arv')
axes[0].set_ylim(0, max(counts.values) + 1)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.05, str(v), ha='center', fontweight='bold')
axes[0].tick_params(axis='x', labelsize=9)

# Sektordiagramm
nonzero = counts[counts > 0]
axes[1].pie(
    nonzero.values,
    labels=nonzero.index,
    autopct='%1.0f%%',
    colors=['#e74c3c', '#e67e22', '#9b59b6'][:len(nonzero)],
    startangle=90
)
axes[1].set_title('Vigade jaotus (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('vigade_analyys.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graafik salvestatud: vigade_analyys.png')

## 4. Järeldused

| Vahesamm | Vigade arv | % vigastest |
|---|---|---|
| Metaandmete filtreerimine | 1 | 17% |
| RAG vektorotsing | 1 | 17% |
| LLM genereerimine | 4 | 67% |

**Peamised tähelepanekud:**

1. **LLM genereerimine on kõige suurem probleemide allikas (67% vigadest).** Mudel kaldus ignoreerima RAG konteksti ja hallutsineerima väliseid ressursse (Coursera, freeCodeCamp, kokakolid jne), eriti siis, kui päring ei sobinud hästi saadaolevate kursustega.

2. **Metaandmete filtreerimine põhjustas 1 vea (17%).** Liiga range filter (EAP=1) jättis andmestiku peaaegu tühjaks, kuid LLM vastas ikkagi väljamõeldud kursustega.

3. **RAG vektorotsing põhjustas 1 vea (17%).** Masinõppe päringu puhul leiti RAGiga ebasobivad kursused (arvutiarhitektuur, biomeditsiinitehnika), mis ei vastanud kasutaja tegelikule vajadusele.

**Soovitused parandamiseks:**
- Lisada süsteemiviipa karm piirang: "Ära soovita ühtegi kursust, mida pole antud nimekirjas"
- Kui filtreerimise järel jääb alla 3 kursuse, hoiatada kasutajat ja laiendada filtreid automaatselt
- Kaaluda reranking-meetodeid RAG otsingu täpsuse parandamiseks

## 5. Testjuhtumid
**Autor: Stepan Sõtšov**

Minimaalselt 5 testjuhtu koos oodatavate unique_ID-dega:

In [ ]:
testjuhtumid = [
    {
        'nr': 1,
        'päring': 'tahan õppida andmeteadust',
        'filtrid': 'Semester: sügis, EAP: 6, Õppeaste: bakalaureuseõpe',
        'oodatavad_ID': ['SVUH.00.059', 'SVNC.00.307'],
        'põhjendus': 'Andmepädevus ja Andmebaasisüsteemid on otseselt andmeteadusega seotud'
    },
    {
        'nr': 2,
        'päring': 'tahan õppida veebiarendust',
        'filtrid': 'Filtrid puuduvad',
        'oodatavad_ID': ['LTAT.03.015', 'MTAT.03.297'],
        'põhjendus': 'Veebilehtede loomine (edasijõudnutele) on otseselt veebiarenduse kursused'
    },
    {
        'nr': 3,
        'päring': 'kvantatarvutused ja kvantfüüsika',
        'filtrid': 'Filtrid puuduvad',
        'oodatavad_ID': ['MTAT.07.024', 'LOFY.04.073', 'LTFY.04.012'],
        'põhjendus': 'Kvantkrüptograafia, Kvantmehaanika ja Kvantarvutuse alused on otseselt seotud'
    },
    {
        'nr': 4,
        'päring': 'tahan õppida programmeerimist inglise keeles',
        'filtrid': 'Keel: inglise keel, Õppeaste: bakalaureuseõpe',
        'oodatavad_ID': ['LTAT.03.001_1', 'LOTI.05.100'],
        'põhjendus': 'Programmeerimise kursused inglise keeles bakalaureuseõppes'
    },
    {
        'nr': 5,
        'päring': 'tahan õppida masinõpet',
        'filtrid': 'Semester: sügis, EAP: 6, Õppeaste: bakalaureuseõpe',
        'oodatavad_ID': ['LTAT.03.001_1', 'LTMS.00.081', 'LOTI.05.100'],
        'põhjendus': 'Programmeerimine ja matemaatika on masinõppe eeldusained'
    }
]

test_df = pd.DataFrame(testjuhtumid)
test_df.columns = ['Nr', 'Päring', 'Filtrid', 'Oodatavad unique_ID-d', 'Põhjendus']
test_df = test_df.set_index('Nr')
print('=== TESTJUHTUMID ===')
test_df